# 折线图实战：解析 JSONP 数据，画三国确诊人数对比

---

**这个案例的价值在哪？**

它把三件事串在了一起，是一次完整的"数据 → 图表"流程：

```
读文件 → 剥掉 JSONP 外壳 → 解析成字典 → 从嵌套结构里取数 → 画折线图
```

**学习目标**

1. 认识 JSONP 格式，知道为什么要"剥壳"
2. 学会**先探索结构、再取数据**的调试方法（这是处理陌生数据的关键技能）
3. 会用 `re` 正则表达式稳健地剥离 JSONP 包装
4. 画多系列折线图并处理"数据点太多"的问题

**数据说明**

原始笔记读的是桌面资料目录下的文件，那个目录已经不存在了。
现在数据放在本 notebook 同级的 `data/折线图数据/` 下，
结构与原来**完全一致**，只是内容是合成的练习数据（详见 `data/README.md`）。


---
## 一、先解决"数据在哪"这个问题

相对路径是相对于**内核启动时的当前目录**，不是相对于 notebook 文件。
从项目根目录启动、从 notebook 目录启动，结果会不一样。

下面写一个小的查找函数，兼容几种常见的启动位置 ——
这也是处理"路径找不到"问题的通用思路：**多试几个候选位置**。


In [ ]:
import os
from pathlib import Path


def data_path(rel):
    """在若干个候选目录下寻找 data/<rel>，返回第一个存在的路径"""
    cwd = Path.cwd()
    candidates = [
        cwd,                                   # 内核就在 09.数据可视化/ 目录下
        cwd / '09.数据可视化',                          # 内核在 项目根目录 下
        cwd / 'src' / 'pythonlearnproject' / '09.数据可视化',
        cwd.parent / '09.数据可视化',                   # 内核在某个子目录下
        cwd.parent,
    ]
    for base in candidates:
        p = base / 'data' / rel
        if p.exists():
            return p
    raise FileNotFoundError(
        f'找不到 data/{rel}\n当前目录是 {cwd}\n'
        '请确认在 可视化/ 或 项目根目录 下启动 Jupyter'
    )


print('当前工作目录：', os.getcwd())
print('示例数据目录：', data_path('折线图数据'))
print('目录里的文件：', [f.name for f in data_path('折线图数据').iterdir()])


---
## 二、先看看"原始数据"长什么样

**处理陌生数据的第一原则：先看，不要猜。**

很多人一上来就写 `json.loads()`，结果报一堆 `JSONDecodeError`，
根本原因是不知道数据外面还包了一层壳。


In [ ]:
us_file = data_path('折线图数据/美国.txt')

with open(us_file, 'r', encoding='utf-8') as f:
    raw = f.read()

print('文件总长度：', len(raw))
print('--- 开头 80 个字符 ---')
print(raw[:80])
print('--- 结尾 40 个字符 ---')
print(raw[-40:])


**看到了吗？** 数据长这样：

```
jsonp_1629344292311_69436({...真正的 JSON...})
    ^^^^^^^^^^^^^^^^^^^^^^                    ^
      这是函数名（外壳）                     这是右括号
```

这就是 **JSONP**（JSON with Padding）：

- 它是一种"用 `<script>` 标签跨域取数据"的老技巧
- 服务器返回的不是纯 JSON，而是"**一个函数调用**"，括号里才是真数据
- 浏览器加载这段脚本时会执行那个函数，从而把数据交给你

**Python 的 `json.loads()` 只认纯 JSON**，所以必须先把 `函数名(` 和末尾的 `)` 去掉。

### 直接 `json.loads` 会怎样？


In [ ]:
import json

try:
    json.loads(raw)
except json.JSONDecodeError as e:
    print('果然失败了 ->', e)
    print('这就是不剥壳的后果。')


---
## 三、剥壳的两种写法

### 写法 1：硬编码函数名（原始笔记的做法，能用但不稳）


In [ ]:
# 原始笔记里是这样写的：把完整的函数名 + "(" 直接替换成空字符串
us_data_v1 = raw.replace('jsonp_1629344292311_69436(', '')
us_data_v1 = us_data_v1[:-2]        # 去掉结尾的 ")"

print('剥壳后前 60 个字：', us_data_v1[:60])
us_dict_v1 = json.loads(us_data_v1)
print('解析成功，顶层键：', list(us_dict_v1.keys()))


**这个写法有什么问题？**

`jsonp_1629344292311_69436` 里那串数字是**时间戳+随机数，每次请求都不一样**。
也就是说这份代码**只对这三次特定的下载有效**，换成新的数据文件，函数名变了，代码就废了。

而且 `[:-2]` 是"砍掉最后两个字符"，**它之所以能砍对，纯属运气**：

```python
文件内容：  ...}")  <- 最后是右括号，然后有一个换行符
去掉函数名后： ...}  +  ")" + "\n"
[:-2] 砍掉末尾两个字符： ")" 和 "\n"   <- 刚好砍对
```

如果这个文件**没有结尾换行符**，`[:-2]` 就会把 `)` 和 `}` 一起砍掉，
结果是 JSON 少了一个闭合大括号，报错：

```
json.decoder.JSONDecodeError: Expecting ',' delimiter
```

这是一个很典型的"**依赖了数据格式的偶然细节**"的坑：
代码在特定文件上能跑，换个文件就崩，而且报错信息还指向一个看起来完全正常的位置。

### 写法 2：用正则表达式（推荐，稳健）


In [ ]:
import re


def parse_jsonp(text):
    """从 JSONP 文本里提取出真正的 JSON 字符串

    兼容这些形式：
        jsonp_123({...})
        callback({...});
        jsonp_123({...})\\n
        {...}                （本身就是纯 JSON）
    """
    text = text.strip()

    # 如果本身以 { 开头，说明就是纯 JSON，直接返回
    if text.startswith('{') or text.startswith('['):
        return text

    # 正则解释：
    #   ^[^(]*      开头到第一个左括号之前的所有内容（就是函数名）
    #   \(          一个左括号
    #   (.*)        中间全部内容（贪婪匹配，会吃到最后一个右括号）
    #   \)[\s;]*$   结尾的右括号，后面可以有空白和分号
    #   re.S        让 . 能匹配换行符
    match = re.match(r'^[^(]*\((.*)\)[\s;]*$', text, re.S)
    if not match:
        raise ValueError('不符合 JSONP 或 JSON 格式')
    return match.group(1)


us_data = parse_jsonp(raw)
print('剥壳后前 60 个字：', us_data[:60])
us_dict = json.loads(us_data)
print('解析成功！')

# 用同一套代码处理另外两个国家，完全不需要知道函数名是什么
jp_dict = json.loads(parse_jsonp(data_path('折线图数据/日本.txt').read_text(encoding='utf-8')))
in_dict = json.loads(parse_jsonp(data_path('折线图数据/印度.txt').read_text(encoding='utf-8')))
print('三个国家都解析完成')


---
## 四、探索结构：不要猜键名，一级一级打印

拿到字典后，**先看它长什么样**。这是处理陌生数据最实用的一招。

我一般用一个递归的小函数，把结构"打印成树"：


In [ ]:
def show_structure(obj, indent=0, max_depth=4, max_items=3):
    """把嵌套的 JSON 结构打印成缩进的树形，方便快速看清层级"""
    pad = '  ' * indent
    if indent > max_depth:
        print(pad + '...（太深了，省略）')
        return

    if isinstance(obj, dict):
        for i, (k, v) in enumerate(obj.items()):
            if i >= max_items and len(obj) > max_items:
                print(pad + f'... 还有 {len(obj) - max_items} 个键')
                break
            preview = f'  ({type(v).__name__})'
            if isinstance(v, (str, int, float)):
                preview = f' = {repr(v)[:60]}'
            elif isinstance(v, list):
                preview = f'  (list, 长度 {len(v)})'
            print(pad + f'{k}{preview}')
            if isinstance(v, (dict, list)):
                show_structure(v, indent + 1, max_depth, max_items)
    elif isinstance(obj, list):
        if obj:
            print(pad + f'[0] ->')
            show_structure(obj[0], indent + 1, max_depth, max_items)


print('=== 美国数据的结构 ===')
show_structure(us_dict)


**看清结构了！** 数据路径是：

```
us_dict
 └─ 'data'          (list)
     └─ [0]        (dict)
         └─ 'trend'          (dict)
             ├─ 'updateDate'  (list)  <- 日期，给 x 轴用
             └─ 'list'        (list)
                  └─ [0]['data']  (list)  <- 数值，给 y 轴用
```

所以取数的代码就是：

```python
us_dict['data'][0]['trend']['updateDate']       # x 轴
us_dict['data'][0]['trend']['list'][0]['data']  # y 轴
```


In [ ]:
us_trend = us_dict['data'][0]['trend']
print('键名有：', list(us_trend.keys()))
print('日期有', len(us_trend['updateDate']), '个，前 3 个：', us_trend['updateDate'][:3])
print('数值有', len(us_trend['list'][0]['data']), '个，前 3 个：', us_trend['list'][0]['data'][:3])


---
## 五、正式画图：三国对比折线图


In [ ]:
from pyecharts.charts import Line
from pyecharts import options as opts


def get_axis_data(d):
    """从一个国家的数据字典里取出 (日期列表, 数值列表)

    这是一个"小工具函数"的典型用法：把重复的取值逻辑封起来，
    处理三个国家时就不用把同一段路径写三遍了。
    """
    trend = d['data'][0]['trend']
    dates = trend['updateDate']
    values = trend['list'][0]['data']
    return dates, values


us_x, us_y = get_axis_data(us_dict)
jp_x, jp_y = get_axis_data(jp_dict)
in_x, in_y = get_axis_data(in_dict)

# 只取 2020 年的数据（原始笔记里截取到下标 314）
N = 314

line = Line()
line.add_xaxis(us_x[:N])          # x 轴只加一次，三个国家共用

line.add_yaxis(
    '美国确诊人数',
    us_y[:N],
    # 【关键】数据点有 300 多个，如果每个点都显示数值标签，图会糊成一片
    label_opts=opts.LabelOpts(is_show=False),
    is_smooth=True,
)
line.add_yaxis(
    '日本确诊人数',
    jp_y[:N],
    label_opts=opts.LabelOpts(is_show=False),
    is_smooth=True,
)
line.add_yaxis(
    '印度确诊人数',
    in_y[:N],
    label_opts=opts.LabelOpts(is_show=False),
    is_smooth=True,
)

line.set_global_opts(
    title_opts=opts.TitleOpts(
        title='2020 年美日印三国确诊人数对比',
        pos_left='center',
        pos_bottom='1%',
    ),
    legend_opts=opts.LegendOpts(pos_top='5%'),
    yaxis_opts=opts.AxisOpts(name='累计确诊人数'),
    tooltip_opts=opts.TooltipOpts(trigger='axis'),
    # x 轴标签太多会重叠，用 DataZoom 让用户自己拖
    datazoom_opts=opts.DataZoomOpts(),
)

line.render('三国确诊人数对比.html')
print('已生成 三国确诊人数对比.html')


---
## 六、如果 x 轴标签太密怎么办？

314 个日期全铺在 x 轴上一定看不清。有三种处理办法：

| 办法 | 做法 | 适合 |
|---|---|---|
| 抽稀标签 | 只看每 10 个中的一个 | 想让全貌一目了然 |
| 加 DataZoom | 用户自己拖 | 想保留查看细节的能力 |
| 旋转标签 | `AxisLabelOpts(rotate=45)` | 标签是文字而不是日期 |


In [ ]:
line2 = Line()
# 每 10 个点取 1 个日期做标签，但数值还是全部保留
sampled_dates = [d if i % 10 == 0 else '' for i, d in enumerate(us_x[:N])]

line2.add_xaxis(sampled_dates)
line2.add_yaxis('美国确诊人数', us_y[:N], label_opts=opts.LabelOpts(is_show=False))
line2.set_global_opts(
    title_opts=opts.TitleOpts(title='x 轴标签抽稀示例', pos_left='center'),
    xaxis_opts=opts.AxisOpts(
        name='日期',
        axislabel_opts=opts.LabelOpts(rotate=45),    # 标签旋转 45 度
    ),
    datazoom_opts=opts.DataZoomOpts(),
)
line2.render('标签抽稀示例.html')
print('已生成 标签抽稀示例.html')


---
## 七、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 不剥 JSONP 外壳直接 `json.loads` | `JSONDecodeError` | 先剥壳（用 `parse_jsonp`） |
| 硬编码 JSONP 函数名 | 换一份数据就失效 | 用正则提取 |
| `[:-2]` 砍尾 | 少砍/多砍，解析失败 | 用 `re.match` 精确匹配 |
| 反复调用 `add_xaxis` | x 轴被覆盖 | x 轴只加一次 |
| 数据点太多还显示标签 | 图表糊成一片 | `label_opts=opts.LabelOpts(is_show=False)` |
| 读文件忘了 `encoding='utf-8'` | 中文乱码 | 显式指定编码 |
| 硬编码绝对路径 | 换电脑就跑不通 | 用 `data_path()` 之类的定位函数 |
| 上来就写取数代码 | 一路 `KeyError` | **先 `show_structure` 看清层级** |
| 忘了 `close()` 文件 | 文件被占用 | 用 `with` 语句 |

---

## 八、动手练习

**练习 1**：`show_structure()` 只打印了前 3 个键。改造它，让它支持一个
`max_items=999` 参数把所有键都打印出来。

**练习 2**：给三国对比图加上"标题居中在顶部"、图例放在左下角、
并且只显示美国一条线（另外两条用 `is_selected=False` 默认隐藏）。
提示：`add_yaxis(..., is_selected=False)`

**练习 3**：算出三个国家第 314 天的确诊人数，用柱状图对比。

**练习 4**：把 `parse_jsonp` 改造得更健壮：如果文本里根本找不到括号，
就抛出一个带说明的 `ValueError`（现在的版本已经做了，读一遍它的实现）。

**练习 5**：把这套流程套到一份你自己的 JSON 数据上：
先 `show_structure` 看清结构，再取数画图。

---

**答案思路（先自己做）**

```python
# 练习 3
from pyecharts.charts import Bar

bar = Bar()
bar.add_xaxis(['美国', '日本', '印度'])
bar.add_yaxis('第314天确诊人数', [us_y[N-1], jp_y[N-1], in_y[N-1]])
bar.set_global_opts(title_opts=opts.TitleOpts(title='三国第314天确诊人数对比'))
bar.render('第314天对比.html')
```
